# A · 러너 — **seed 0,1 / GPU 0,1**

GPU 를 2장씩 쓰므로 **창 두 개**로 나눠 돌린다. 이 창은 **A 몫**만 맡는다.

| 창 | seed | GPU |
|---|---|---|
| **A (이 노트북)** | **[0, 1]** | **[0, 1]** |
| B (다른 노트북 `B_run_seed23.ipynb`) | [2, 3] | [2, 3] |

두 창이 **다른 GPU 를 쓰므로 동시에 띄워도 안 밟는다.** 결과는 같은 경로에 쌓여
리포트(`09`/`10`/`18`)에서 **자동으로 seed 4개로 합쳐진다**.

> **GPU 가 2장뿐인 노드**라면: 이 창(A)을 먼저 끝내고, 그다음 B 노트북에서
> `GPUS = [0, 1]` 로 바꿔서 돌리면 된다(같은 GPU 를 순차 재사용).

각 셀은 **그룹 하나**다. 필요한 것만 순서대로 실행하면 되고, 끊겨도 재실행하면 이어간다
(학습=resume, eval=끝난 run skip).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

PART = 'A'
SEEDS, GPUS = cf.part(PART)        # A → seeds [0, 1], GPU [0, 1]
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

MAIN  = cf.MAIN_SIM                # 'insertion' (긴 horizon, 메인)
SHORT = cf.SHORT_SIM               # 'transfer'  (짧은 앵커)

print('PART :', PART, '| seeds:', SEEDS, '| GPU:', GPUS)
print('보이는 GPU:', cf.v23.available_gpus())
print('학습: %s step (lr 고정, 학습중 eval %s)' % (f'{cf.STEPS:,}', cf.v23.EVAL_FREQ))
print('eval : %s ckpt x %d rep x %d ep' % (f'{cf.CKPT_STEP:,}', len(REPS), N_EP))
print()
print('그룹: ours=%s  acm=%s  baseline=%s  ablation=%s'
      % (cf.GROUP_OURS, cf.GROUP_ACM, cf.GROUP_BASELINE, cf.GROUP_ABLATION))

## 1) insertion · **ours** — 학습

In [ ]:
cf.run_training(cf.GROUP_OURS, SEEDS, task=MAIN, gpus=GPUS)

## 2) insertion · **ours** — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_OURS, SEEDS, REPS, task=MAIN, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_OURS, SEEDS, REPS, task=MAIN, n_episodes=N_EP)

## 3) insertion · **acm**(대조군) — 학습

In [ ]:
cf.run_training(cf.GROUP_ACM, SEEDS, task=MAIN, gpus=GPUS)

## 4) insertion · **acm** — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_ACM, SEEDS, REPS, task=MAIN, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_ACM, SEEDS, REPS, task=MAIN, n_episodes=N_EP)

## 5) transfer · **ours** — 학습

In [ ]:
cf.run_training(cf.GROUP_OURS, SEEDS, task=SHORT, gpus=GPUS)

## 6) transfer · **ours** — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_OURS, SEEDS, REPS, task=SHORT, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_OURS, SEEDS, REPS, task=SHORT, n_episodes=N_EP)

## 7) transfer · **acm** — 학습

In [ ]:
cf.run_training(cf.GROUP_ACM, SEEDS, task=SHORT, gpus=GPUS)

## 8) transfer · **acm** — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_ACM, SEEDS, REPS, task=SHORT, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_ACM, SEEDS, REPS, task=SHORT, n_episodes=N_EP)

## 9) insertion · **baseline**(act·diffusion·smolvla·acm2) — 학습

In [ ]:
cf.run_training(cf.GROUP_BASELINE, SEEDS, task=MAIN, gpus=GPUS)

## 10) insertion · **baseline** + act_te — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_BASELINE + ['act_te'], SEEDS, REPS, task=MAIN, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_BASELINE + ['act_te'], SEEDS, REPS, task=MAIN, n_episodes=N_EP)

## 11) transfer · **baseline** — 학습

In [ ]:
cf.run_training(cf.GROUP_BASELINE, SEEDS, task=SHORT, gpus=GPUS)

## 12) transfer · **baseline** + act_te — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_BASELINE + ['act_te'], SEEDS, REPS, task=SHORT, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_BASELINE + ['act_te'], SEEDS, REPS, task=SHORT, n_episodes=N_EP)

## 13) insertion · **ablation**(carry/BiMamba/MOSAIC 분리) — 학습

In [ ]:
cf.run_training(cf.GROUP_ABLATION, SEEDS, task=MAIN, gpus=GPUS)

## 14) insertion · **ablation** — eval

In [ ]:
cf.run_repeat_evals(cf.GROUP_ABLATION, SEEDS, REPS, task=MAIN, gpus=GPUS, n_episodes=N_EP)
cf.sr_table(cf.GROUP_ABLATION, SEEDS, REPS, task=MAIN, n_episodes=N_EP)

## 상태 확인 (이 창의 seed [0, 1] 만)

In [ ]:
for task in (MAIN, SHORT):
    print(f'\n===== {task} =====')
    for grp in (cf.GROUP_OURS, cf.GROUP_ACM, cf.GROUP_BASELINE, cf.GROUP_ABLATION):
        cf.print_ckpt_status(grp, SEEDS, task)

## 다음
- 두 창(A·B)이 다 끝나면 seed 4개가 모인다 → `09_report_sr` · `10_report_jerk` · `18_report_horizon`
- 효율(학습 불필요): `11_efficiency`
- 그룹 하나만 따로 보고 싶으면 개별 노트북(`01`~`08`, `12`~`17`)을 쓰면 된다.
  이 러너는 그 노트북들을 **seed/GPU 를 쪼개서** 한 번에 돌리는 용도.